In [47]:
import numpy as np
import pandas as pd
from pathlib import Path
 
# BERTopic and its sub-components
from bertopic import BERTopic
from bertopic.representation import KeyBERTInspired
from sklearn.feature_extraction.text import CountVectorizer
 
# UMAP and HDBSCAN — BERTopic uses these internally but we configure them explicitly
# so results are reproducible
from umap import UMAP
from hdbscan import HDBSCAN

In [48]:
df = pd.read_json("../data/processed/final_songs.json")
llm_embeddings = np.load("../data/cache/llm_embedding.npy")
umap_embeddings = np.load("../data/cache/umap_embedding.npy")

In [49]:
umap_model = UMAP(
        n_components=5,       # reduce to 5D before clustering (BERTopic default)
        n_neighbors=15,
        min_dist=0.0,         # tighter clusters for topic discovery
        metric="cosine",
        random_state=42,
    )

hdbscan_model = HDBSCAN(
        min_cluster_size=15,
        min_samples=5,
        metric="euclidean",
        cluster_selection_method="eom",
        prediction_data=True,  # required for soft-clustering / probability scores
    )

# Use unigrams and bigrams; filter very common/rare words
vectorizer_model = CountVectorizer(
    ngram_range=(1,2),
    stop_words="english",
    min_df=1,             # word must appear in at least 1 songs
    max_df=0.95,          # ignore words that appear in >85% of songs (too common)
)

In [50]:
representation_model = KeyBERTInspired()

In [51]:
model = BERTopic(
        umap_model=umap_model,
        hdbscan_model=hdbscan_model,
        vectorizer_model=vectorizer_model,
        representation_model=representation_model,
        top_n_words=10,           # keywords per topic
        verbose=True,
    )

In [52]:
docs = df['preprocessed_lyrics'].fillna("").tolist()

topic_ids, probabilities = model.fit_transform(docs)

2026-03-20 10:59:01,749 - BERTopic - Embedding - Transforming documents to embeddings.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7367.71it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 44/44 [00:05<00:00,  7.62it/s]
2026-03-20 10:59:08,910 - BERTopic - Embedding - Completed ✓
2026-03-20 10:59:08,910 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-03-20 10:59:11,946 - BERTopic - Dimensionality - Completed ✓
2026-03-20 10:59:11,947 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-03-20 10:59:11,980 - BERTopic - Cluster - Completed ✓
2026-03-20 10:59:11,982 - BERTopic - Representation - Fine-tuning topics using representation mo

In [57]:
summary = model.get_document_info(docs)

In [59]:
summary[summary["Representative_document"]==True]

,Document,Topic,Name,Representation,Representative_Docs,Top_n_words,Probability,Representative_document
11,Bad tattoos on leather-tanned skin Jesus Chris...,1,1_making feel_roses black_nanananananana nanan...,"[making feel, roses black, nanananananana nana...",[Bad tattoos on leather-tanned skin Jesus Chri...,making feel - roses black - nanananananana nan...,0.765020,True
131,You choke my throat with words of wonder You m...,1,1_making feel_roses black_nanananananana nanan...,"[making feel, roses black, nanananananana nana...",[Bad tattoos on leather-tanned skin Jesus Chri...,making feel - roses black - nanananananana nan...,0.790171,True
145,Purple and scarred It's the color that will be...,1,1_making feel_roses black_nanananananana nanan...,"[making feel, roses black, nanananananana nana...",[Bad tattoos on leather-tanned skin Jesus Chri...,making feel - roses black - nanananananana nan...,0.479478,True
159,We just wanna party all day We just wanna part...,0,0_gimme love_girl girl_love gimme_party girl,"[gimme love, girl girl, love gimme, party girl...",[Sometimes I'll let voices in my head (Da-da-d...,gimme love - girl girl - love gimme - party gi...,1.000000,True
183,"You think you got it all right, all right But ...",-1,-1_wanna die_babe_im gonna_feel feel,"[wanna die, babe, im gonna, feel feel, wanna t...","[La-la-la-la, la-la-la-la, la La-la-la-la, la-...",wanna die - babe - im gonna - feel feel - wann...,0.000000,True
310,"La-la-la-la, la-la-la-la, la La-la-la-la, la-l...",-1,-1_wanna die_babe_im gonna_feel feel,"[wanna die, babe, im gonna, feel feel, wanna t...","[La-la-la-la, la-la-la-la, la La-la-la-la, la-...",wanna die - babe - im gonna - feel feel - wann...,0.000000,True
695,"(Three and four and) Ba-ba-da, ba-ba-da-ba-da ...",0,0_gimme love_girl girl_love gimme_party girl,"[gimme love, girl girl, love gimme, party girl...",[Sometimes I'll let voices in my head (Da-da-d...,gimme love - girl girl - love gimme - party gi...,1.000000,True
1053,Sometimes I'll let voices in my head (Da-da-da...,0,0_gimme love_girl girl_love gimme_party girl,"[gimme love, girl girl, love gimme, party girl...",[Sometimes I'll let voices in my head (Da-da-d...,gimme love - girl girl - love gimme - party gi...,1.000000,True
1339,"La, la, la, la, la, la, la La, la, la, la, la,...",-1,-1_wanna die_babe_im gonna_feel feel,"[wanna die, babe, im gonna, feel feel, wanna t...","[La-la-la-la, la-la-la-la, la La-la-la-la, la-...",wanna die - babe - im gonna - feel feel - wann...,0.000000,True


In [58]:
summary

,Document,Topic,Name,Representation,Representative_Docs,Top_n_words,Probability,Representative_document
0,"Girl, it's so confusing sometimes to be a girl...",0,0_gimme love_girl girl_love gimme_party girl,"[gimme love, girl girl, love gimme, party girl...",[Sometimes I'll let voices in my head (Da-da-d...,gimme love - girl girl - love gimme - party gi...,1.000000,False
1,I only threw this party for you Only threw thi...,0,0_gimme love_girl girl_love gimme_party girl,"[gimme love, girl girl, love gimme, party girl...",[Sometimes I'll let voices in my head (Da-da-d...,gimme love - girl girl - love gimme - party gi...,0.992577,False
2,I went my own way and I made it I'm your favou...,0,0_gimme love_girl girl_love gimme_party girl,"[gimme love, girl girl, love gimme, party girl...",[Sometimes I'll let voices in my head (Da-da-d...,gimme love - girl girl - love gimme - party gi...,1.000000,False
3,Mm Oh I guess the apple don't fall far from th...,0,0_gimme love_girl girl_love gimme_party girl,"[gimme love, girl girl, love gimme, party girl...",[Sometimes I'll let voices in my head (Da-da-d...,gimme love - girl girl - love gimme - party gi...,1.000000,False
4,I don't wanna share the space I don't wanna fo...,0,0_gimme love_girl girl_love gimme_party girl,"[gimme love, girl girl, love gimme, party girl...",[Sometimes I'll let voices in my head (Da-da-d...,gimme love - girl girl - love gimme - party gi...,1.000000,False
...,...,...,...,...,...,...,...,...
1383,"Got a job, got a crib, got a mind of my own, h...",0,0_gimme love_girl girl_love gimme_party girl,"[gimme love, girl girl, love gimme, party girl...",[Sometimes I'll let voices in my head (Da-da-d...,gimme love - girl girl - love gimme - party gi...,1.000000,False
1384,"I was a liar, I gave in to the fire I know I s...",0,0_gimme love_girl girl_love gimme_party girl,"[gimme love, girl girl, love gimme, party girl...",[Sometimes I'll let voices in my head (Da-da-d...,gimme love - girl girl - love gimme - party gi...,1.000000,False
1385,"Yeah Hey old friends, let's look back on the c...",0,0_gimme love_girl girl_love gimme_party girl,"[gimme love, girl girl, love gimme, party girl...",[Sometimes I'll let voices in my head (Da-da-d...,gimme love - girl girl - love gimme - party gi...,1.000000,False
1386,"Mmm, I, mmm Share my life Take me for what I a...",0,0_gimme love_girl girl_love gimme_party girl,"[gimme love, girl girl, love gimme, party girl...",[Sometimes I'll let voices in my head (Da-da-d...,gimme love - girl girl - love gimme - party gi...,1.000000,False
